# Building a Memory-Enhanced Conversational Agent

## Setup and Imports

First, we'll import the necessary modules and set up our environment.

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


C:\Users\sk335\OneDrive\Documents\Coding\PythonProjects\langchain-and-langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Memory Stores
We'll create stores for both short-term (chat history) and long-term memory.

In [2]:
chat_store = {}
long_term_store = {}

def get_chat_history(session_id: str):
    if session_id not in chat_store:
        chat_store[session_id] = ChatMessageHistory()
    return chat_store[session_id]

def update_long_term_memory(session_id: str, input: str, output: str):
    if session_id not in long_term_store:
        long_term_store[session_id] = []
    if len(input) > 20:
        long_term_store[session_id].append(f"user said: {input}")
    if len(long_term_store[session_id]) > 5:
        long_term_store[session_id] = long_term_store[session_id][-5:]

def get_long_term_memory(session_id: str):
    return ". ".join(long_term_store.get(session_id, []))

# Prompt Template
We'll create a prompt template that includes both short-term and long-term memory.

In [3]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the information from long-term-memory if relevant."),
    # Use a LangChain variable, not an f-string
    ("system", "Long term memory: {long_term_memory}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Conversational Chain
Now, we'll set up the conversational chain with message history.

In [4]:
chain = prompt | llm
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    # Make sure this matches the MessagesPlaceholder variable name!
    history_messages_key="history",
)

# Chat Function
We'll create a function to handle chat interactions, including updating long-term memory.

In [5]:
def chat(input_text: str, session_id: str):
    long_term_mem = get_long_term_memory(session_id)
    response = chain_with_history.invoke(
        {"input": input_text, "long_term_memory": long_term_mem},
        config={"configurable": {"session_id": session_id}}
    )
    update_long_term_memory(session_id, input_text, response.content)
    return response.content

In [6]:
session_id = "user_123"

print("AI:", chat("Hello! My name is Alice.", session_id))
print("AI:", chat("What's the weather like today?", session_id))
print("AI:", chat("I love sunny days.", session_id))
print("AI:", chat("Do you remember my name?", session_id))

AI: Hello Alice! It's nice to meet you. How can I help you today?
AI: I apologize, but I don't have access to real-time information, including current weather conditions. My capabilities don't extend to fetching live data.

To find out the weather, I recommend checking a weather app, a weather website, or your local news!
AI: That's wonderful! Sunny days can really lift your spirits. There's something special about the warmth and brightness they bring.
AI: Yes, I do! Your name is Alice.


# Review Memory
Let's review the conversation history and long-term memory.

In [7]:
print("Conversation History:")
for message in chat_store[session_id].messages:
    print(f"{message.type}: {message.content}")

print("\nLong-term Memory:")
print(get_long_term_memory(session_id))

Conversation History:
human: Hello! My name is Alice.
ai: Hello Alice! It's nice to meet you. How can I help you today?
human: What's the weather like today?
ai: I apologize, but I don't have access to real-time information, including current weather conditions. My capabilities don't extend to fetching live data.

To find out the weather, I recommend checking a weather app, a weather website, or your local news!
human: I love sunny days.
ai: That's wonderful! Sunny days can really lift your spirits. There's something special about the warmth and brightness they bring.
human: Do you remember my name?
ai: Yes, I do! Your name is Alice.

Long-term Memory:
user said: Hello! My name is Alice.. user said: What's the weather like today?. user said: Do you remember my name?
